# Traduction français ↔ baoulé : préparation, adaptation et évaluation

Ce notebook réalise la première brique texte du cahier des charges : traduction
français ↔ baoulé (IA-02) et évaluation automatique (IA-05).
Il ne réalise pas encore la détection de langue, le dioula, le dialogue par LLM,
l’API ou la voix. La relecture par des locuteurs reste indispensable.

## Parcours d’exécution
1. Installer les dépendances selon `INSTALLATION_PC.md` et sélectionner le noyau `.venv`.
2. Exécuter configuration, préparation et contrôle des données.
3. Tokeniser, entraîner les adaptateurs LoRA, puis évaluer le test réservé.
4. Pour utiliser un modèle déjà entraîné, suivre la dernière section dans un noyau neuf.

Le token `aka_Latn` est un repère technique d’adaptation pour le baoulé,
pas une garantie de couverture linguistique. Les sources et licences doivent être
documentées avant toute diffusion du corpus ou du modèle.


In [1]:
# Installation facultative : laisser False si le guide a déjà été suivi.
# Installer PyTorch CUDA depuis le terminal conformément à INSTALLATION_PC.md.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    from pathlib import Path
    requirements_file = Path('requirements-pc-rtx5070.txt')
    if not requirements_file.is_file():
        raise FileNotFoundError('Ouvrir le notebook depuis la racine du projet.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)])
    print('Redémarrer le noyau avant de continuer.')


## 1. Configuration et environnement

Les chemins sont relatifs au dossier du projet. La préparation fonctionne sur CPU ;
l’entraînement nécessite CUDA par défaut. Aucun téléchargement de modèle n’a lieu
avant la tokenisation. Le premier téléchargement nécessite Internet.


In [2]:
from pathlib import Path
import os
# Cache local pour éviter les erreurs de permissions sur le dossier utilisateur.
os.environ.setdefault("HF_HOME", str(Path.cwd() / ".cache" / "huggingface"))
import random
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import set_seed

# Le notebook doit être ouvert depuis le dossier contenant le corpus.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'datasets').is_dir():
    raise FileNotFoundError('Ouvrir le notebook depuis la racine du projet contenant datasets/.')

DATA_DIR = PROJECT_DIR / 'datasets'
MODEL_DIR = PROJECT_DIR / 'models' / 'nllb-baoule-lora'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
FR_TOKEN = 'fra_Latn'
BAOULE_TOKEN = 'aka_Latn'  # repère d’adaptation, pas un token baoulé natif
CORPUS_PATH = DATA_DIR / 'corpus_fr_baoule_adjoumani.csv'
SEED = 42
MAX_LENGTH = 128
set_seed(SEED)
import sys
print('Python :', sys.executable)
print('Projet :', PROJECT_DIR)
print('Corpus :', CORPUS_PATH)
print('PyTorch :', torch.__version__, '| Transformers :', transformers.__version__)
print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM totale (Gio) :', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
    print('CUDA PyTorch :', torch.version.cuda)
else:
    print('Préparation et tests possibles sur CPU. Entraînement désactivé sans GPU par défaut.')


c:\Users\testo\OneDrive\Documents\GitHub\assistant-ia-langues-locales\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python : c:\Users\testo\OneDrive\Documents\GitHub\assistant-ia-langues-locales\.venv\Scripts\python.exe
Projet : c:\Users\testo\OneDrive\Documents\GitHub\assistant-ia-langues-locales
Corpus : c:\Users\testo\OneDrive\Documents\GitHub\assistant-ia-langues-locales\datasets\corpus_fr_baoule_adjoumani.csv
PyTorch : 2.7.1+cu128 | Transformers : 4.44.2
CUDA : True
GPU : NVIDIA GeForce RTX 5070 Laptop GPU
VRAM totale (Gio) : 8.0
CUDA PyTorch : 12.8


## 2. Nettoyage et séparation du corpus

Source : `datasets/corpus_fr_baoule_adjoumani.csv`, colonnes `french`, `baoule`
et éventuellement `source_url`. Les fichiers d’origine restent conservés.
Les paires vides et les doublons normalisés sont retirés sans supprimer les accents.
Les phrases françaises ou baoulé communes sont regroupées dans un seul jeu.
Les proportions visées sont 80 % entraînement, 10 % validation et 10 % test ;
les groupes peuvent modifier ces proportions. Les quasi-doublons sémantiques
et la qualité des traductions nécessitent encore une vérification humaine.


In [3]:
import csv
import json
import unicodedata
from collections import defaultdict

corpus_path = CORPUS_PATH
if not corpus_path.is_file():
    raise FileNotFoundError(f'Corpus introuvable : {corpus_path}')

def clean_text(text):
    return ' '.join(unicodedata.normalize('NFC', str(text)).split())

def phrase_key(text):
    return clean_text(text).casefold().replace('’', "'")

with corpus_path.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
if not rows or not {'french', 'baoule'}.issubset(rows[0]):
    raise ValueError('Colonnes attendues : french, baoule (source_url facultative).')
cleaned, seen = [], set()
empty_rows = duplicate_rows = 0
for row in rows:
    fr, bci = clean_text(row.get('french') or ''), clean_text(row.get('baoule') or '')
    pair = (phrase_key(fr), phrase_key(bci))
    if not fr or not bci:
        empty_rows += 1
        continue
    if pair in seen:
        duplicate_rows += 1
        continue
    seen.add(pair)
    cleaned.append({'french': fr, 'baoule': bci, 'source_url': row.get('source_url', '')})

# Composantes connexes : une phrase FR OU baoulé partagée impose le même jeu.
parent = list(range(len(cleaned)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
owners = {}
for i, row in enumerate(cleaned):
    for lang in ('french', 'baoule'):
        key = (lang, phrase_key(row[lang]))
        if key in owners:
            parent[find(i)] = find(owners[key])
        else:
            owners[key] = i
groups = defaultdict(list)
for i, row in enumerate(cleaned):
    groups[find(i)].append(row)
blocks = list(groups.values())
random.Random(SEED).shuffle(blocks)
if len(blocks) < 3:
    raise ValueError('Corpus insuffisant pour créer trois jeux indépendants.')
splits = {'train': [], 'validation': [], 'test': []}
target = max(1, round(len(cleaned) * 0.1))
# Réserver au moins un groupe à chaque jeu, même pour un petit corpus.
for name, block in zip(('test', 'validation', 'train'), blocks[:3]):
    splits[name].extend(block)
for block in blocks[3:]:
    name = 'test' if len(splits['test']) < target else ('validation' if len(splits['validation']) < target else 'train')
    splits[name].extend(block)
for name, data in splits.items():
    if not data:
        raise ValueError(f'Jeu {name} vide : vérifier les groupes du corpus.')
for lang in ('french', 'baoule'):
    keys = {name: {phrase_key(r[lang]) for r in data} for name, data in splits.items()}
    for a, b in [('train', 'test'), ('train', 'validation'), ('validation', 'test')]:
        assert keys[a].isdisjoint(keys[b]), (lang, a, b)
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name, data in splits.items():
    pd.DataFrame(data).to_csv(DATA_DIR / f'baoule_v2_{name}.csv', index=False)
    print(name, len(data))
print('Aucun recouvrement de phrase normalisée entre les trois jeux.')

print(f'Lignes source : {len(rows)} ; vides : {empty_rows} ; doublons : {duplicate_rows}')
print(f'Paires conservées : {len(cleaned)} ; groupes indépendants : {len(blocks)}')
display(pd.DataFrame(splits['train']).head())


train 1612
validation 201
test 201
Aucun recouvrement de phrase normalisée entre les trois jeux.
Lignes source : 2043 ; vides : 0 ; doublons : 29
Paires conservées : 2014 ; groupes indépendants : 1928


,french,baoule,source_url
0,Pourquoi devrions- nousprêterattention à la dé...,Ngue ti yɛ ɔ fata kɛ mmoja’n i su mmla nga Ɲan...,https://fr.glosbe.com/fr/bci/pr%C3%AAter
1,Jésus a déclaré : “ Tu dis : ‘ Je suis riche e...,Zezi seli be kɛ: ‘An se kɛ an ti aɲanbeunfuɛ y...,https://fr.glosbe.com/fr/bci/pauvre%20%3B%20pi...
2,🐕 Chien,🐕 Alua,"https://aistudio.google.com/, https://translat..."
3,Quelbonheurque l’amourdeDieu s’étende à tous c...,"Nga m’ɔ ti ndɛ fɛ’n, yɛle kɛ Ɲanmiɛn klo sran ...",https://fr.glosbe.com/fr/bci/Homme%20de%20bonheur
4,💃 Femme qui danse,💃 Bla n'ga su si able,"https://aistudio.google.com/, https://translat..."


## 3. Tokenisation bidirectionnelle

Chaque paire donne un exemple français → baoulé et un exemple baoulé → français.
Le tokenizer reçoit explicitement les langues source et cible. Le padding dynamique
est ignoré dans la perte. Vérifier le nombre d’exemples tronqués ; augmenter
`MAX_LENGTH` si nécessaire, en tenant compte de la mémoire GPU.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
for token in (FR_TOKEN, BAOULE_TOKEN):
    assert tokenizer.convert_tokens_to_ids(token) != tokenizer.unk_token_id, token

class TranslationDataset(Dataset):
    def __init__(self, rows):
        self.items = []
        truncated = 0
        for row in rows:
            for source, target, src_token, tgt_token in (
                (row['french'], row['baoule'], FR_TOKEN, BAOULE_TOKEN),
                (row['baoule'], row['french'], BAOULE_TOKEN, FR_TOKEN),
            ):
                tokenizer.src_lang = src_token
                tokenizer.tgt_lang = tgt_token
                truncated += int(len(tokenizer(source)['input_ids']) > MAX_LENGTH or
                                 len(tokenizer(text_target=target)['input_ids']) > MAX_LENGTH)
                item = tokenizer(source, truncation=True, max_length=MAX_LENGTH)
                item['labels'] = tokenizer(text_target=target, truncation=True,
                                           max_length=MAX_LENGTH)['input_ids']
                self.items.append(item)
        print(f'{len(self.items)} exemples bidirectionnels ; {truncated} exemples tronqués '
              f'({100 * truncated / max(1, len(self.items)):.1f} %).')
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

train_dataset = TranslationDataset(splits['train'])
validation_dataset = TranslationDataset(splits['validation'])


c:\Users\testo\OneDrive\Documents\GitHub\assistant-ia-langues-locales\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


3224 exemples bidirectionnels ; 20 exemples tronqués (0.6 %).
402 exemples bidirectionnels ; 6 exemples tronqués (1.5 %).


## 4. Fine-tuning : étapes et suivi en direct

1. **Chargement** : téléchargement de NLLB au premier lancement, puis lecture du cache.
   Cette phase précède la barre d’entraînement et peut prendre du temps.
2. **Adaptateurs LoRA** : le modèle de base est gelé ; seuls de petits ensembles de
   paramètres des projections d’attention sont entraînés. Le nombre de paramètres
   entraînables est affiché avant le démarrage.
3. **Apprentissage** : chaque mini-lot produit une prédiction, une perte, puis des
   gradients. Avec un batch de 1 et une accumulation de 16, une mise à jour utilise
   normalement 16 exemples par GPU (le dernier groupe peut être incomplet).
   Une **étape** affichée correspond à une mise à jour, pas à une phrase.
   Une **époque** correspond à un passage sur tout le jeu d’entraînement.
4. **Validation** : à la fin de chaque époque, le modèle calcule sa perte sur des
   exemples séparés, sans apprendre dessus. Le jeu de test n’intervient pas ici.
5. **Checkpoint** : après chaque époque, une sauvegarde permet la reprise.
   Les deux derniers checkpoints sont conservés, ainsi que le meilleur si nécessaire.
   Le meilleur modèle selon la perte de validation est rechargé en fin d’entraînement.
6. **Export** : sauvegarde des adaptateurs, du tokenizer, des métadonnées et du journal.

### Lire les indicateurs pendant l’exécution

La barre native affiche les étapes effectuées, le temps et une estimation du temps
restant. Un résumé textuel s’affiche dès la première mise à jour, puis toutes les
5 mises à jour ; la validation et les sauvegardes ont aussi leurs messages.
Si la barre ne s’affiche pas dans VS Code, les résumés textuels restent disponibles.

| Indicateur | Interprétation |
|---|---|
| Progression | Pourcentage des mises à jour prévues ; ce n’est pas une mesure de qualité. |
| `loss` | Perte moyenne sur l’intervalle de journalisation ; une baisse est souhaitable, avec des fluctuations normales. |
| `eval_loss` | Perte sur la validation ; utilisée pour choisir le meilleur checkpoint. |
| `learning_rate` | Taux d’apprentissage courant, ajusté par le scheduler. |
| `grad_norm` | Norme des gradients rapportée par le Trainer ; aide à repérer une instabilité. |
| Temps restant estimé | Projection à partir des étapes de cette session ; varie avec validation, sauvegardes et charge GPU. |

Si `loss` baisse mais `eval_loss` augmente durablement, le modèle peut surapprendre.
Ces pertes ne sont ni un pourcentage de réussite ni les scores BLEU/chrF : ceux-ci
sont calculés ensuite sur le test, et doivent être complétés par une relecture humaine.

Les événements sont enregistrés au fil de l’exécution dans
`models/nllb-baoule-lora/training_progress.csv` (une nouvelle session ajoute ses lignes).
`trainer_log_history.json` contient l’historique exporté à la fin du run.
En cas de mémoire insuffisante, redémarrer le noyau avant de modifier la configuration.
Changer `MODEL_DIR` pour conserver séparément plusieurs expériences.


In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


import csv
import time
from datetime import datetime, timezone
from transformers import TrainerCallback

class TrainingProgressCallback(TrainerCallback):
    """Suivi texte et CSV sans dépendance à des widgets Jupyter."""
    fields = ['session', 'event', 'step', 'total_steps', 'epoch', 'progress_pct',
              'elapsed_s', 'eta_s', 'loss', 'eval_loss', 'learning_rate', 'grad_norm']

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()
        self.initial_step = state.global_step
        self.session = datetime.now(timezone.utc).isoformat()
        self.csv_path = Path(args.output_dir) / 'training_progress.csv'
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'Apprentissage : {state.max_steps} mises à jour prévues au total ; '
              f'reprise à l’étape {state.global_step}.', flush=True)
        self.record('start', state)

    def record(self, event, state, logs=None):
        elapsed = time.monotonic() - self.started
        completed = state.global_step - self.initial_step
        eta = elapsed / completed * max(0, state.max_steps - state.global_step) if completed > 0 else None
        row = dict(session=self.session, event=event, step=state.global_step,
                   total_steps=state.max_steps, epoch=state.epoch,
                   progress_pct=100 * state.global_step / max(1, state.max_steps),
                   elapsed_s=round(elapsed, 1), eta_s=round(eta, 1) if eta is not None else '')
        row.update({key: (logs or {})[key] for key in self.fields if key in (logs or {})})
        new_file = not self.csv_path.exists() or self.csv_path.stat().st_size == 0
        with self.csv_path.open('a', encoding='utf-8', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=self.fields)
            if new_file:
                writer.writeheader()
            writer.writerow(row)
        return row

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        row = self.record('validation' if 'eval_loss' in logs else 'log', state, logs)
        values = ' | '.join(f'{k}={logs[k]:.5g}' for k in
                            ('loss', 'eval_loss', 'learning_rate', 'grad_norm') if k in logs)
        eta_text = f"{row['eta_s'] / 60:.1f} min" if row['eta_s'] != '' else 'en cours de calcul'
        epoch = state.epoch or 0
        print(f"Étape {state.global_step}/{state.max_steps} ({row['progress_pct']:.1f} %) | "
              f"époque {epoch:.2f} | écoulé {row['elapsed_s'] / 60:.1f} min | "
              f"reste estimé {eta_text} | {values}", flush=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        print('Passage d’apprentissage terminé : validation puis sauvegarde.', flush=True)

    def on_prediction_step(self, args, state, control, **kwargs):
        # La barre native affiche la progression des lots de validation.
        pass

    def on_save(self, args, state, control, **kwargs):
        self.record('checkpoint', state)
        print(f'Checkpoint sauvegardé : {Path(args.output_dir) / f"checkpoint-{state.global_step}"}', flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        self.record('end', state)
        print('Apprentissage terminé ; export des adaptateurs et des journaux.', flush=True)


ALLOW_CPU_TRAINING = False
if not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
    raise RuntimeError('GPU CUDA non détecté. Vérifier le pilote NVIDIA, la version CUDA de PyTorch et le noyau sélectionné. ALLOW_CPU_TRAINING permet un essai CPU très lent.')
if 'model' in globals() or 'trainer' in globals():
    raise RuntimeError('Modèle déjà chargé : redémarrer le noyau avant de relancer cette cellule.')

print('1/6 — Chargement du modèle de base (téléchargement ou cache)…', flush=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('2/6 — Configuration des adaptateurs LoRA…', flush=True)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=8, lora_alpha=16,
    lora_dropout=0.05, target_modules=['q_proj', 'v_proj'],
))
model.enable_input_require_grads()
model.print_trainable_parameters()
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
args = Seq2SeqTrainingArguments(
    output_dir=str(MODEL_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-4,
    num_train_epochs=3,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, save_total_limit=2,
    logging_strategy='steps', logging_steps=5, logging_first_step=True,
    disable_tqdm=False, report_to='none', seed=SEED,
    dataloader_num_workers=0,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_dataset,
    eval_dataset=validation_dataset, data_collator=collator,
    tokenizer=tokenizer, callbacks=[TrainingProgressCallback()],
)
# Après interruption, indiquer le chemin d’un checkpoint existant pour reprendre.
RESUME_FROM_CHECKPOINT = None
print('3/6 — Démarrage ; 4/6 validation et 5/6 checkpoint à chaque époque.', flush=True)
result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
print('6/6 — Export final…', flush=True)
trainer.save_model(str(MODEL_DIR))
trainer.save_state()
(MODEL_DIR / 'trainer_log_history.json').write_text(
    json.dumps(trainer.state.log_history, ensure_ascii=False, indent=2), encoding='utf-8')
tokenizer.save_pretrained(str(MODEL_DIR))
metadata = {
    'base_model': MODEL_NAME, 'fr_token': FR_TOKEN, 'baoule_token': BAOULE_TOKEN,
    'seed': SEED, 'max_length': MAX_LENGTH,
    'split_sizes': {name: len(data) for name, data in splits.items()},
    'metrics': result.metrics,
    'corpus_path': str(CORPUS_PATH.relative_to(PROJECT_DIR)),
    'corpus_sha256': __import__('hashlib').sha256(CORPUS_PATH.read_bytes()).hexdigest(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
}
(MODEL_DIR / 'training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Adaptateurs et tokenizer sauvegardés :', MODEL_DIR)


## 5. Traduction et évaluation finale

Exécuter cette section après l’entraînement ou après le rechargement des adaptateurs.
La génération impose le token de la langue cible. Le test réservé sert à calculer
BLEU et chrF dans chaque sens et à exporter les prédictions pour une relecture.
Les scores du modèle adapté ne prouvent pas une amélioration sans comparaison
avec le modèle de base sur le même test. Ne pas ajuster les paramètres sur le test.


In [ ]:
model.eval()
model.config.use_cache = True

def translate(text, direction='fr-bci'):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Le texte doit être une chaîne non vide.')
    if direction not in ('fr-bci', 'bci-fr'):
        raise ValueError('direction attendue : fr-bci ou bci-fr')
    src, tgt = (FR_TOKEN, BAOULE_TOKEN) if direction == 'fr-bci' else (BAOULE_TOKEN, FR_TOKEN)
    tokenizer.src_lang, tokenizer.tgt_lang = src, tgt
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_LENGTH).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                                max_new_tokens=MAX_LENGTH, num_beams=2)
    return tokenizer.batch_decode(output, skip_special_tokens=True)[0]

print(translate('Bonjour, comment vas-tu ?'))


In [ ]:
import sacrebleu

scores = {}
for direction, source_col, target_col in [('fr-bci', 'french', 'baoule'), ('bci-fr', 'baoule', 'french')]:
    predictions, references, records = [], [], []
    for i, row in enumerate(splits['test']):
        prediction = translate(row[source_col], direction)
        predictions.append(prediction)
        references.append(row[target_col])
        records.append({'source': row[source_col], 'reference': row[target_col], 'prediction': prediction})
        if (i + 1) % 50 == 0:
            print(direction, i + 1, '/', len(splits['test']))
    bleu = sacrebleu.metrics.BLEU()
    chrf = sacrebleu.metrics.CHRF()
    scores[direction] = {'BLEU': bleu.corpus_score(predictions, [references]).score,
                         'chrF': chrf.corpus_score(predictions, [references]).score,
                         'BLEU_signature': str(bleu.get_signature()),
                         'chrF_signature': str(chrf.get_signature())}
    pd.DataFrame(records).to_csv(MODEL_DIR / f'predictions_{direction}.csv', index=False)
    print(direction, scores[direction])
(MODEL_DIR / 'test_metrics.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')


## 6. Utiliser les adaptateurs sauvegardés dans une nouvelle session

Dans un noyau neuf : exécuter la configuration (section 1), puis le bloc ci-dessous
en activant `LOAD_SAVED_MODEL`, puis la cellule définissant `translate()` (section 5).
La préparation et la tokenisation ne sont pas nécessaires pour traduire.
Pour recalculer les scores, exécuter également la préparation du corpus avec la
même source et la même graine que lors de l’entraînement.

Le dossier sauvegardé contient des adaptateurs : le modèle de base reste nécessaire.
Les prochains travaux du cahier des charges concernent la validation linguistique,
le dioula, puis l’intégration au pipeline conversationnel, à l’API et à la voix.


In [ ]:
LOAD_SAVED_MODEL = False  # True uniquement dans un noyau neuf.
if LOAD_SAVED_MODEL:
    from peft import PeftModel
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    if 'model' in globals() or 'trainer' in globals():
        raise RuntimeError('Redémarrer le noyau pour éviter de charger deux modèles.')
    if not (MODEL_DIR / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Adaptateurs absents de {MODEL_DIR}. Entraîner le modèle d’abord.')
    import json
    metadata_path = MODEL_DIR / 'training_metadata.json'
    if metadata_path.is_file():
        saved_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        MODEL_NAME = saved_metadata['base_model']
        FR_TOKEN = saved_metadata['fr_token']
        BAOULE_TOKEN = saved_metadata['baoule_token']
        MAX_LENGTH = saved_metadata['max_length']
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = PeftModel.from_pretrained(base, MODEL_DIR)
    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    print('Modèle rechargé. Exécuter la cellule définissant translate().')
